<a href="https://colab.research.google.com/github/maclandrol/cours-ia-med/blob/master/06_Segmentation_MedSAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06. Segmentation d'Images Médicales avec MedSAM

**Enseignant:** Emmanuel Noutahi, PhD

---

## Objectifs

- Utiliser MedSAM pour segmenter des structures médicales
- Guider la segmentation avec des points
- Calculer des mesures cliniques (aire, diamètre)

## Qu'est-ce que la segmentation?

Délimiter précisément des structures dans une image:
- **Oncologie**: Mesurer une tumeur
- **Cardiologie**: Volume cardiaque
- **Chirurgie**: Planification pré-opératoire

**MedSAM**: Segmentation interactive guidée par points (1-2 min vs 30-60 min manuel)

## 1. Installation

In [ ]:
!pip install -q transformers torch torchvision pillow matplotlib numpy scikit-image

In [ ]:
import torch
from transformers import SamModel, SamProcessor
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage import measure
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

## 2. Chargement de MedSAM

In [ ]:
print("Chargement de MedSAM...")

model = SamModel.from_pretrained("flaviagiammarino/medsam-vit-base")
processor = SamProcessor.from_pretrained("flaviagiammarino/medsam-vit-base")

model.to(device)
model.eval()

print("MedSAM chargé!")

## 3. Chargement d'une Image

Uploadez une image médicale (CT, IRM, radiographie, échographie, etc.)

In [ ]:
from google.colab import files

uploaded = files.upload()

if uploaded:
    img_path = list(uploaded.keys())[0]
    image = Image.open(img_path).convert("RGB")
    
    plt.figure(figsize=(8, 8))
    plt.imshow(image)
    plt.title("Image à Segmenter")
    plt.axis('off')
    plt.show()
    
    print(f"\nDimensions: {image.size}")
    print("\nCliquez mentalement sur la structure à segmenter.")
    print("Notez les coordonnées (x, y) approximatives.")
else:
    print("Aucune image uploadée.")

## 4. Segmentation Interactive

Guidez MedSAM avec des points:
- **Points positifs**: Structure à segmenter
- **Points négatifs** (optionnel): Zones à exclure

In [ ]:
def segment(image, input_points, input_labels):
    """Segmente avec MedSAM."""
    inputs = processor(
        image,
        input_points=[[input_points]],
        input_labels=[[input_labels]],
        return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    masks = processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs["original_sizes"].cpu(),
        inputs["reshaped_input_sizes"].cpu()
    )
    
    return masks[0][0][0].numpy()

# MODIFIEZ ces coordonnées selon votre image
# Format: [x, y] pour chaque point
points_positifs = [250, 250]  # Point dans la structure à segmenter
points_negatifs = []  # Optionnel: [[x1, y1], [x2, y2]]

# Préparer les inputs
if points_negatifs:
    all_points = [points_positifs] + points_negatifs
    all_labels = [1] + [0] * len(points_negatifs)  # 1=positif, 0=négatif
else:
    all_points = [points_positifs]
    all_labels = [1]

# Segmentation
if 'image' in locals():
    print("Segmentation en cours...")
    mask = segment(image, all_points, all_labels)
    print("Terminé!")
else:
    print("Uploadez d'abord une image.")

## 5. Visualisation

In [ ]:
if 'mask' in locals():
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Image + point
    axes[0].imshow(image)
    for i, (pt, label) in enumerate(zip(all_points, all_labels)):
        color = 'green' if label == 1 else 'red'
        marker = 'o' if label == 1 else 'x'
        axes[0].plot(pt[0], pt[1], marker=marker, color=color, markersize=12, 
                    markeredgewidth=2, markeredgecolor='white')
    axes[0].set_title("Image + Points")
    axes[0].axis('off')
    
    # Masque
    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title("Masque de Segmentation")
    axes[1].axis('off')
    
    # Superposition
    axes[2].imshow(image)
    mask_overlay = np.zeros((*mask.shape, 4))
    mask_overlay[mask] = [1, 0, 0, 0.4]  # Rouge transparent
    axes[2].imshow(mask_overlay)
    axes[2].set_title("Superposition")
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

## 6. Mesures Cliniques

In [ ]:
def compute_metrics(mask, pixel_size_mm=0.5):
    """Calcule mesures cliniques."""
    area_pixels = np.sum(mask)
    area_mm2 = area_pixels * (pixel_size_mm ** 2)
    
    # Diamètre équivalent
    diameter_mm = 2 * np.sqrt(area_mm2 / np.pi)
    
    # Contours
    contours = measure.find_contours(mask.astype(float), 0.5)
    perimeter_mm = len(contours[0]) * pixel_size_mm if contours else 0
    
    # Circularité
    circularity = (4 * np.pi * area_mm2) / (perimeter_mm ** 2) if perimeter_mm > 0 else 0
    
    return {
        'area_mm2': area_mm2,
        'diameter_mm': diameter_mm,
        'perimeter_mm': perimeter_mm,
        'circularity': circularity
    }

if 'mask' in locals():
    metrics = compute_metrics(mask)
    
    print("\nMESURES CLINIQUES")
    print("="*40)
    print(f"Aire:        {metrics['area_mm2']:.1f} mm²")
    print(f"Diamètre:    {metrics['diameter_mm']:.1f} mm")
    print(f"Périmètre:   {metrics['perimeter_mm']:.1f} mm")
    print(f"Circularité: {metrics['circularity']:.2f}")
    print("="*40)
    print("\nCircularité: 1.0 = cercle parfait")

## 7. Raffinement (Optionnel)

Si la segmentation n'est pas satisfaisante:
1. Retournez à la section 4
2. Ajoutez des points négatifs pour exclure des zones
3. Ou ajustez les points positifs

In [ ]:
# Exemple avec raffinement
# points_positifs = [250, 250]
# points_negatifs = [[300, 300], [200, 300]]  # Zones à exclure

print("Modifiez les points dans la section 4 et ré-exécutez.")

## 8. Points Clés

### Ce que vous avez appris:
- Segmenter des structures avec MedSAM
- Guider avec points positifs/négatifs
- Calculer mesures cliniques

### Applications:
- **Oncologie**: Mesure et suivi de tumeurs
- **Cardiologie**: Volumes cardiaques
- **Chirurgie**: Planification pré-opératoire

### Limites:
- Qualité d'image importante
- Validation par expert nécessaire
- Peut nécessiter plusieurs points

### Approche recommandée:
1. Segmentation rapide par MedSAM (1-2 min)
2. Correction manuelle si nécessaire
3. Validation par radiologue

**Gain de temps: 70-80% vs segmentation manuelle complète**